In [3]:
import pandas as pd
from pandas.tseries.offsets import DateOffset
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import griddata
import datetime as dt
from pathlib import Path
import os
from tqdm import tqdm

In [4]:
# Find nearest index
def find_index(array, x):
    if array.ndim == 1:
        idx = np.argmin(np.abs(array - x))
    elif array.ndim == 2:
        idx = np.unravel_index(np.argmin(np.abs(array - x)), array.shape)
    else:
        raise ValueError("Unsupported array dimensions for find_index function.")
    return idx

In [5]:
#Read in BC wet dep timeseries files
result_dir = Path('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Timeseries_atm_h2')
bc_files_to_open = ['FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.a2x_BCPHIWET.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.bc_a1SFWET.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.bc_a4SFWET.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.bc_c1SFWET.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.bc_c4SFWET.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.PRECC.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.PRECL.nc']

file_paths = [result_dir / file_name for file_name in bc_files_to_open]
nc_daily_bc = xr.open_mfdataset(file_paths, combine='nested')

#read in POM files
daily_pom_to_open = 'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.pom_*SFWET.nc'
nc_daily_pom = xr.open_mfdataset(str(result_dir/daily_pom_to_open),combine='nested')

#read in SOA
soa_result_dir = Path('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Timeseries_atm_SOAvars/')
daily_soa_to_open = 'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.*SFWET.nc'
nc_daily_soa = xr.open_mfdataset(str(soa_result_dir/daily_soa_to_open),combine='nested')

##merge SOA and POM files together
nc_daily_oc = xr.merge([nc_daily_soa, nc_daily_pom, nc_daily_bc])
nc_daily_oc

<xarray.Dataset>
Dimensions:       (time: 7671, lat: 192, lon: 288)
Coordinates:
  * lat           (lat) float64 -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon           (lon) float64 0.0 1.25 2.5 3.75 ... 355.0 356.2 357.5 358.8
  * time          (time) datetime64[ns] 2002-01-01 2002-01-02 ... 2023-01-01
Data variables: (12/31)
    soa1_a1SFWET  (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    soa1_a2SFWET  (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    soa1_c1SFWET  (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    soa1_c2SFWET  (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    soa2_a1SFWET  (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    soa2_a2SFWET  (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    ...            ...
    bc_a1SFWET    (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    bc_a4SFWET    (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    bc_c1SFWET    (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    bc_c4SFWET    (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    PRECC         (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    PRECL         (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              FCnudged_f09.mam.Murray.Apr11_01.2002_2023.001
    logname:           demurray
    host:              derecho1
    initial_file:      /glade/campaign/acom/acom-climate/UTLS/shawnh/archive/...
    topography_file:   /glade/campaign/cesm/cesmdata/inputdata/atm/cam/topo/f...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [6]:
# Calculate BC_wet_sum, and DOC = SOA + POM + BC
nc_daily_oc['BC_wet_sum'] = (nc_daily_oc['bc_a1SFWET'] + nc_daily_oc['bc_a4SFWET'] + nc_daily_oc['bc_c1SFWET'] 
                                + nc_daily_oc['bc_c4SFWET']) * -1

nc_daily_oc['SOA_POM_wet_sum'] = (nc_daily_oc['soa1_a1SFWET'] + nc_daily_oc['soa1_a2SFWET'] + nc_daily_oc['soa1_c1SFWET'] 
                             + nc_daily_oc['soa1_c2SFWET'] + nc_daily_oc['soa2_a1SFWET'] + nc_daily_oc['soa2_a2SFWET'] 
                             + nc_daily_oc['soa2_c1SFWET'] +  nc_daily_oc['soa2_c2SFWET'] + nc_daily_oc['soa3_a1SFWET'] 
                             + nc_daily_oc['soa3_a2SFWET'] + nc_daily_oc['soa3_c1SFWET'] + nc_daily_oc['soa3_c2SFWET'] 
                             + nc_daily_oc['soa4_a1SFWET'] + nc_daily_oc['soa4_a2SFWET'] + nc_daily_oc['soa4_c1SFWET'] 
                             + nc_daily_oc['soa4_c2SFWET'] + nc_daily_oc['soa5_a1SFWET'] + nc_daily_oc['soa5_a2SFWET'] 
                             + nc_daily_oc['soa5_c1SFWET'] + nc_daily_oc['soa5_c2SFWET'] + nc_daily_oc['pom_a1SFWET']
                             + nc_daily_oc['pom_a4SFWET'] + nc_daily_oc['pom_c1SFWET'] + nc_daily_oc['pom_c4SFWET']) * -1

nc_daily_oc['SOA_POM_BC_wet_sum'] = nc_daily_oc['BC_wet_sum'] + nc_daily_oc['SOA_POM_wet_sum']

nc_daily_oc['PRECC_mm'] = nc_daily_oc['PRECC']* (86400*1000) # seconds to days and m to mm = mm/d
nc_daily_oc['PRECL_mm'] = nc_daily_oc['PRECL']* (86400*1000)
nc_daily_oc['PREC_tot_mm'] = nc_daily_oc['PRECC_mm']  + nc_daily_oc['PRECL_mm']

# Perform unit conversions
conversions = 86400 * 1000000  # seconds to days and kg to mg per m2
nc_daily_oc = nc_daily_oc.apply(lambda x: x * conversions if x.name in ['BC_wet_sum', 'SOA_POM_wet_sum', 'SOA_POM_BC_wet_sum'] else x)
nc_daily_oc  

<xarray.Dataset>
Dimensions:             (lat: 192, lon: 288, time: 7671)
Coordinates:
  * lat                 (lat) float64 -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon                 (lon) float64 0.0 1.25 2.5 3.75 ... 356.2 357.5 358.8
  * time                (time) datetime64[ns] 2002-01-01 ... 2023-01-01
Data variables: (12/37)
    soa1_a1SFWET        (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    soa1_a2SFWET        (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    soa1_c1SFWET        (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    soa1_c2SFWET        (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    soa2_a1SFWET        (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    soa2_a2SFWET        (time, lat, lon) float32 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    ...                  ...
    BC_wet_sum          (time, lat, lon) float64 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    SOA_POM_wet_sum     (time, lat, lon) float64 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    SOA_POM_BC_wet_sum  (time, lat, lon) float64 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    PRECC_mm            (time, lat, lon) float64 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    PRECL_mm            (time, lat, lon) float64 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>
    PREC_tot_mm         (time, lat, lon) float64 dask.array<chunksize=(7671, 192, 288), meta=np.ndarray>

In [7]:
#Select cells that correspond to NADP sites: read in NADP lat/long and apply the find nearest function
pathData = '/glade/u/home/demurray/External File Uploads/DOC data/'
os.chdir(pathData)
doc_bc_df = pd.read_csv('DOC_BC_wetdep_compiled.csv')
doc_bc_sites = doc_bc_df[['siteId', 'latitude', 'longitude']].drop_duplicates()
doc_bc_sites

,siteId,latitude,longitude
0,SR,44.483333,-72.166667
1,NH02,43.950000,-71.733333
18,CO90,40.050000,-105.583000
531,TF,43.108000,-70.950000
704,OR10,44.266640,-122.177000
...,...,...,...
4164,NM07,35.778800,-106.266000
4165,NE99,41.059200,-100.746400
4195,CO01,38.117700,-103.316000
4196,FL23,30.110600,-84.990200


In [8]:
# Only select the cells in .nc that correspond to an NADP site lat/long
subset_list = []
progress_bar = tqdm(total=len(doc_bc_sites))
for index, row in doc_bc_sites.iterrows():
    lat = row['latitude']
    lon = 360-(row['longitude']*-1)   # longitude in the model is positive and based on 360 degrees.
    lat_idx = find_index(nc_daily_oc['lat'].values, lat)   # right now we are doing a 'find nearest' calculation, should probably interpolate across grid cell and have exact coordinates represented?
    lon_idx = find_index(nc_daily_oc['lon'].values, lon)
    subset = nc_daily_oc.isel(lat=lat_idx, lon=lon_idx)
    subset['siteId'] = row['siteId']  # Add 'siteId' as a new coordinate/index
    subset = subset.assign_coords(siteId=row['siteId'])
    subset_list.append(subset)
    progress_bar.update(1)
progress_bar.close()

# Concatenate the list of subsets into a new xarray dataset
nc_daily_nadp = xr.concat(subset_list, dim='siteId')
nc_daily_nadp

100%|██████████| 214/214 [00:02<00:00, 94.07it/s] 


<xarray.Dataset>
Dimensions:             (siteId: 214, time: 7671)
Coordinates:
    lat                 (siteId) float64 44.76 43.82 40.05 ... 38.17 29.69 37.23
    lon                 (siteId) float64 287.5 288.8 255.0 ... 256.2 275.0 253.8
  * time                (time) datetime64[ns] 2002-01-01 ... 2023-01-01
  * siteId              (siteId) <U4 'SR' 'NH02' 'CO90' ... 'CO01' 'FL23' 'CO91'
Data variables: (12/37)
    soa1_a1SFWET        (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    soa1_a2SFWET        (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    soa1_c1SFWET        (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    soa1_c2SFWET        (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    soa2_a1SFWET        (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    soa2_a2SFWET        (siteId, time) float32 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    ...                  ...
    BC_wet_sum          (siteId, time) float64 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    SOA_POM_wet_sum     (siteId, time) float64 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    SOA_POM_BC_wet_sum  (siteId, time) float64 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    PRECC_mm            (siteId, time) float64 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    PRECL_mm            (siteId, time) float64 dask.array<chunksize=(1, 7671), meta=np.ndarray>
    PREC_tot_mm         (siteId, time) float64 dask.array<chunksize=(1, 7671), meta=np.ndarray>

In [7]:
'''
#Add a loop that for each siteId it writes a file to timeseries
output_directory = '/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Daily_Timeseries_NADPsites'

unique_site_ids = nc_daily_subset['siteId'].values

# Initialize tqdm
pbar = tqdm(unique_site_ids, desc="Writing subset files")

# Iterate over each unique siteId
for site_id in pbar:
    # Subset the dataset for the current siteId
    subset_ds = nc_daily_subset.where(nc_daily_subset['siteId'] == site_id, drop=True)
    
    # Construct the filename
    filename = f'{site_id}_DailyTimeseries_WetDep_BC.nc'
    
    # Write the subsetted dataset to the specified directory
    output_path = os.path.join(output_directory, filename)
    subset_ds.to_netcdf(output_path)
    
    # Update tqdm description
    pbar.set_description(f"Writing subset files: {filename}")
'''

'\n#Add a loop that for each siteId it writes a file to timeseries\noutput_directory = \'/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Daily_Timeseries_NADPsites\'\n\nunique_site_ids = nc_daily_subset[\'siteId\'].values\n\n# Initialize tqdm\npbar = tqdm(unique_site_ids, desc="Writing subset files")\n\n# Iterate over each unique siteId\nfor site_id in pbar:\n    # Subset the dataset for the current siteId\n    subset_ds = nc_daily_subset.where(nc_daily_subset[\'siteId\'] == site_id, drop=True)\n    \n    # Construct the filename\n    filename = f\'{site_id}_DailyTimeseries_WetDep_BC.nc\'\n    \n    # Write the subsetted dataset to the specified directory\n    output_path = os.path.join(output_directory, filename)\n    subset_ds.to_netcdf(output_path)\n    \n    # Update tqdm description\n    pbar.set_description(f"Writing subset files: {filename}")\n'

In [9]:
#Turn nc_daily_nadp xarray into a pandas dataframe with similar attributes to the NADP dataset
mod_nadp = nc_daily_nadp.to_dataframe().reset_index()
mod_nadp = mod_nadp.rename(columns = {'time': 'model_time','SOA_POM_BC_wet_sum': 'Mod_SOA_POM_BC_mgm2', 'BC_wet_sum': 'Mod_BC_mgm2', 'SOA_POM_wet_sum': 'Mod_SOA_POM_mgm2'})

#IMPORTANT STEP: change the time to one day prior (model writes time at end of current day)
mod_nadp['time'] = pd.to_datetime(mod_nadp['model_time'],format='%Y-%m-%d')+DateOffset(days=-1)

mod_nadp.head(10)

,siteId,model_time,lat,lon,soa1_a1SFWET,soa1_a2SFWET,soa1_c1SFWET,soa1_c2SFWET,soa2_a1SFWET,soa2_a2SFWET,...,bc_c4SFWET,PRECC,PRECL,Mod_BC_mgm2,Mod_SOA_POM_mgm2,Mod_SOA_POM_BC_mgm2,PRECC_mm,PRECL_mm,PREC_tot_mm,time
0,SR,2002-01-01,44.764398,287.5,-2.761495e-19,-2.390102e-20,-1.632851e-13,-1.194081e-14,-2.136568e-19,-1.898661e-20,...,-0.0,0.0,5.117070e-10,3.599695e-03,0.062647,0.066247,0.0,0.044211,0.044211,2001-12-31
1,SR,2002-01-02,44.764398,287.5,-5.162339e-19,-2.213282e-19,-3.469285e-13,-2.537138e-14,-4.394023e-19,-1.965019e-19,...,-0.0,0.0,8.861764e-10,1.474170e-02,0.191238,0.205980,0.0,0.076566,0.076566,2002-01-01
2,SR,2002-01-03,44.764398,287.5,-3.001594e-19,-1.250460e-19,-9.082579e-14,-1.129406e-14,-2.194070e-19,-9.208264e-20,...,-0.0,0.0,3.391328e-10,3.081979e-03,0.050878,0.053960,0.0,0.029301,0.029301,2002-01-02
3,SR,2002-01-04,44.764398,287.5,0.000000e+00,0.000000e+00,-1.623524e-16,-3.530663e-18,0.000000e+00,0.000000e+00,...,0.0,0.0,4.327553e-13,6.060697e-06,0.000079,0.000085,0.0,0.000037,0.000037,2002-01-03
4,SR,2002-01-05,44.764398,287.5,-7.791289e-18,-1.734101e-18,0.000000e+00,0.000000e+00,-5.284411e-18,-1.012547e-18,...,0.0,0.0,1.643673e-09,5.835076e-07,0.000005,0.000005,0.0,0.142013,0.142013,2002-01-04
5,SR,2002-01-06,44.764398,287.5,-1.595407e-16,-2.988449e-18,-4.200651e-14,-3.841167e-15,-1.298241e-16,-1.935127e-18,...,0.0,0.0,3.496329e-09,1.512066e-03,0.018684,0.020196,0.0,0.302083,0.302083,2002-01-05
6,SR,2002-01-07,44.764398,287.5,-1.358359e-15,-1.359497e-17,-2.113774e-12,-4.064994e-14,-1.087270e-15,-9.243841e-18,...,0.0,0.0,3.215727e-08,5.973816e-02,0.805848,0.865586,0.0,2.778388,2.778388,2002-01-06
7,SR,2002-01-08,44.764398,287.5,-1.796949e-15,-1.883858e-17,-5.863426e-13,-3.529584e-14,-1.409682e-15,-1.199216e-17,...,0.0,0.0,7.444969e-08,1.826881e-02,0.211616,0.229885,0.0,6.432453,6.432453,2002-01-07
8,SR,2002-01-09,44.764398,287.5,-2.818391e-19,-7.078272e-20,-1.435735e-15,-2.994858e-16,-2.374859e-19,-4.096852e-20,...,0.0,0.0,1.787460e-10,4.613563e-05,0.000653,0.000699,0.0,0.015444,0.015444,2002-01-08
9,SR,2002-01-10,44.764398,287.5,-1.204602e-15,-5.149841e-17,-2.431526e-12,-7.075751e-14,-1.022604e-15,-4.043747e-17,...,0.0,0.0,6.239301e-08,9.361750e-02,0.988595,1.082213,0.0,5.390756,5.390756,2002-01-09


In [10]:
mod_nadp.columns

Index(['siteId', 'model_time', 'lat', 'lon', 'soa1_a1SFWET', 'soa1_a2SFWET',
       'soa1_c1SFWET', 'soa1_c2SFWET', 'soa2_a1SFWET', 'soa2_a2SFWET',
       'soa2_c1SFWET', 'soa2_c2SFWET', 'soa3_a1SFWET', 'soa3_a2SFWET',
       'soa3_c1SFWET', 'soa3_c2SFWET', 'soa4_a1SFWET', 'soa4_a2SFWET',
       'soa4_c1SFWET', 'soa4_c2SFWET', 'soa5_a1SFWET', 'soa5_a2SFWET',
       'soa5_c1SFWET', 'soa5_c2SFWET', 'pom_a1SFWET', 'pom_a4SFWET',
       'pom_c1SFWET', 'pom_c4SFWET', 'a2x_BCPHIWET', 'bc_a1SFWET',
       'bc_a4SFWET', 'bc_c1SFWET', 'bc_c4SFWET', 'PRECC', 'PRECL',
       'Mod_BC_mgm2', 'Mod_SOA_POM_mgm2', 'Mod_SOA_POM_BC_mgm2', 'PRECC_mm',
       'PRECL_mm', 'PREC_tot_mm', 'time'],
      dtype='object')

In [11]:
#IGNORE THIS CELL IF RUNNING FULL ANALYSES, THIS IS JUST FOR COMPARISON WITH MUSICAV0 RUN
#truncate time
start = dt.datetime.strptime('2016-12-31', '%Y-%m-%d')
end = dt.datetime.strptime('2018-12-31', '%Y-%m-%d')
mod_nadp_sub = mod_nadp.loc[(mod_nadp.time > start) & (mod_nadp.time < end),]

#truncate variables
mod_nadp_sub = mod_nadp_sub[['siteId', 'model_time', 'lat', 'lon', 'PREC_tot_mm', 'Mod_SOA_POM_BC_mgm2', 'time']]

#write file
mod_nadp_sub.to_csv('/glade/u/home/demurray/Murray-NCAR-GVP/MUSICA Analyses/Data outputs/DOCproxy_global20172018_cleaned_nadpsitemerged.csv')

In [9]:
#read in timeseries of: NADP NTN and ensure correct/consistent formatting
doc_bc_df = pd.read_csv('DOC_BC_wetdep_compiled.csv', parse_dates = ['dateOn', 'dateOff'])
nadp_df = doc_bc_df

#Scale concentrations to mg/m2 using precip volume, assuming 1mm rain = 1L/m2
nadp_df['Value_mgm2'] = (nadp_df['Value'] * nadp_df['Precip_mm']) 

#Ensure correct datetime formatting
nadp_df = nadp_df.sort_values(['siteId', 'dateOn'], ascending = True)

#Need to round date because using DAILY data for modelled comparisons
nadp_df['dateOnround'] = nadp_df.dateOn + dt.timedelta(hours=12)
nadp_df['dateOnround'] = pd.to_datetime(nadp_df.dateOnround.dt.strftime('%Y-%m-%d'))
nadp_df['dateOffround'] = nadp_df.dateOff + dt.timedelta(hours=12)
nadp_df['dateOffround'] = pd.to_datetime(nadp_df.dateOffround.dt.strftime('%Y-%m-%d'))

# select relevant columns
nadp_df = nadp_df[['siteId','latitude', 'longitude', 'dateOn', 'dateOff', 'dateOnround', 'dateOffround', 'Variable', 'Value_mgm2', 'Precip_mm']]

nadp_df.head(5)

,siteId,latitude,longitude,dateOn,dateOff,dateOnround,dateOffround,Variable,Value_mgm2,Precip_mm
3847,AB32,57.1894,-111.6406,2020-10-28,2020-11-04,2020-10-28,2020-11-04,rBC_mgL,0.005256,5.84
3869,AB32,57.1894,-111.6406,2020-11-04,2020-11-10,2020-11-04,2020-11-10,rBC_mgL,0.037476,10.41
4087,AB32,57.1894,-111.6406,2020-11-17,2020-11-24,2020-11-17,2020-11-24,rBC_mgL,0.054303,7.87
3757,AB34,55.6214,-111.1727,2020-10-28,2020-11-03,2020-10-28,2020-11-03,rBC_mgL,0.010290,6.86
3862,AB34,55.6214,-111.1727,2020-11-03,2020-11-10,2020-11-03,2020-11-10,rBC_mgL,0.039024,16.26


In [10]:
##Assign sampling intervals to NADP NTN deposition data
nadp_df['SamplingInt'] = pd.Series(dtype='int')
nadp_df['IntTime'] = pd.Series(dtype='int')

sites = nadp_df.siteId.unique()

for i in tqdm(sites, unit = 'sites', total = len(sites), ncols = 100):
    nadp_df.loc[nadp_df.siteId == i,'SamplingInt'] = list(range(0, len(nadp_df.loc[nadp_df.siteId == i]), 1))
    nadp_df.loc[nadp_df.siteId == i, 'IntTime'] = nadp_df.loc[nadp_df.siteId == i, 'dateOffround'] - nadp_df.loc[nadp_df.siteId == i, 'dateOnround']
nadp_df.head(10)

100%|█████████████████████████████████████████████████████████| 210/210 [00:00<00:00, 310.79sites/s]


,siteId,latitude,longitude,dateOn,dateOff,dateOnround,dateOffround,Variable,Value_mgm2,Precip_mm,SamplingInt,IntTime
3847,AB32,57.1894,-111.6406,2020-10-28,2020-11-04,2020-10-28,2020-11-04,rBC_mgL,0.005256,5.84,0.0,7 days 00:00:00
3869,AB32,57.1894,-111.6406,2020-11-04,2020-11-10,2020-11-04,2020-11-10,rBC_mgL,0.037476,10.41,1.0,6 days 00:00:00
4087,AB32,57.1894,-111.6406,2020-11-17,2020-11-24,2020-11-17,2020-11-24,rBC_mgL,0.054303,7.87,2.0,7 days 00:00:00
3757,AB34,55.6214,-111.1727,2020-10-28,2020-11-03,2020-10-28,2020-11-03,rBC_mgL,0.010290,6.86,0.0,6 days 00:00:00
3862,AB34,55.6214,-111.1727,2020-11-03,2020-11-10,2020-11-03,2020-11-10,rBC_mgL,0.039024,16.26,1.0,7 days 00:00:00
4167,AB34,55.6214,-111.1727,2020-11-18,2020-11-25,2020-11-18,2020-11-25,rBC_mgL,0.001218,2.03,2.0,7 days 00:00:00
3849,AB36,57.2592,-111.0386,2020-10-28,2020-11-04,2020-10-28,2020-11-04,rBC_mgL,0.007267,5.59,0.0,7 days 00:00:00
3876,AB36,57.2592,-111.0386,2020-11-04,2020-11-10,2020-11-04,2020-11-10,rBC_mgL,0.014688,4.32,1.0,6 days 00:00:00
4100,AB36,57.2592,-111.0386,2020-11-16,2020-11-24,2020-11-16,2020-11-24,rBC_mgL,0.016764,3.81,2.0,8 days 00:00:00
3905,AK01,65.1550,-147.4910,2020-11-03,2020-11-10,2020-11-03,2020-11-10,rBC_mgL,0.128016,35.56,0.0,7 days 00:00:00


In [11]:
#Write loop to assign sampling intervals to  modelled data frame
mod_nadp['SamplingInt'] = pd.Series(dtype='int') # Add a SamplingInt column to the modelled
sites =nadp_df.siteId.unique() #[0:170]

for i in tqdm(sites, unit = "sites", total = len(sites), ncols = 100):
    sampleInt = nadp_df.loc[nadp_df.siteId==i,'SamplingInt']
    #print(i)
    for j in sampleInt:
       #print(j)
       begDate = pd.Timestamp(nadp_df.loc[(nadp_df.siteId == i) & (nadp_df.SamplingInt == j), 'dateOnround'].item())
       endDate = pd.Timestamp(nadp_df.loc[(nadp_df.siteId == i) & (nadp_df.SamplingInt == j), 'dateOffround'].item())
       #print(endDate)
       mod_nadp.loc[(mod_nadp.siteId == i) & (mod_nadp.time >= begDate) & (mod_nadp.time < endDate), 'SamplingInt'] = j 

mod_nadp.drop_duplicates(inplace = True)
mod_nadp.to_csv('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Daily_Timeseries_NADPsites/DOC_BC_wetdep_Daily_SamplingIntAssigned.csv')
mod_nadp.head(20)

100%|██████████████████████████████████████████████████████████| 210/210 [06:51<00:00,  1.96s/sites]


,siteId,model_time,lat,lon,soa1_a1SFWET,soa1_a2SFWET,soa1_c1SFWET,soa1_c2SFWET,soa2_a1SFWET,soa2_a2SFWET,...,a2x_BCPHIWET,bc_a1SFWET,bc_a4SFWET,bc_c1SFWET,bc_c4SFWET,Mod_BC_mgm2,Mod_SOA_POM_mgm2,Mod_SOA_POM_BC_mgm2,time,SamplingInt
0,SR,2002-01-01,44.764398,287.5,-2.761495e-19,-2.390102e-20,-1.632851e-13,-1.194081e-14,-2.136568e-19,-1.898661e-20,...,4.166313e-14,-6.549075e-20,-5.209951e-19,-4.166255e-14,-0.0,3.599695e-03,0.062647,0.066247,2001-12-31,98.0
1,SR,2002-01-02,44.764398,287.5,-5.162339e-19,-2.213282e-19,-3.469285e-13,-2.537138e-14,-4.394023e-19,-1.965019e-19,...,1.706215e-13,-2.153090e-19,-1.411442e-18,-1.706199e-13,-0.0,1.474170e-02,0.191238,0.205980,2002-01-01,98.0
2,SR,2002-01-03,44.764398,287.5,-3.001594e-19,-1.250460e-19,-9.082579e-14,-1.129406e-14,-2.194070e-19,-9.208264e-20,...,3.567106e-14,-1.115586e-19,-4.834321e-19,-3.567046e-14,-0.0,3.081979e-03,0.050878,0.053960,2002-01-02,99.0
3,SR,2002-01-04,44.764398,287.5,0.000000e+00,0.000000e+00,-1.623524e-16,-3.530663e-18,0.000000e+00,0.000000e+00,...,7.014695e-17,0.000000e+00,0.000000e+00,-7.014695e-17,0.0,6.060697e-06,0.000079,0.000085,2002-01-03,99.0
4,SR,2002-01-05,44.764398,287.5,-7.791289e-18,-1.734101e-18,0.000000e+00,0.000000e+00,-5.284411e-18,-1.012547e-18,...,6.753560e-18,-3.496725e-18,-3.256835e-18,0.000000e+00,0.0,5.835076e-07,0.000005,0.000005,2002-01-04,99.0
5,SR,2002-01-06,44.764398,287.5,-1.595407e-16,-2.988449e-18,-4.200651e-14,-3.841167e-15,-1.298241e-16,-1.935127e-18,...,1.750077e-14,-6.562026e-17,-2.585805e-17,-1.740929e-14,0.0,1.512066e-03,0.018684,0.020196,2002-01-05,99.0
6,SR,2002-01-07,44.764398,287.5,-1.358359e-15,-1.359497e-17,-2.113774e-12,-4.064994e-14,-1.087270e-15,-9.243841e-18,...,6.914139e-13,-5.833371e-16,-1.942585e-16,-6.906363e-13,0.0,5.973816e-02,0.805848,0.865586,2002-01-06,99.0
7,SR,2002-01-08,44.764398,287.5,-1.796949e-15,-1.883858e-17,-5.863426e-13,-3.529584e-14,-1.409682e-15,-1.199216e-17,...,2.114446e-13,-7.270689e-16,-2.458208e-16,-2.104717e-13,0.0,1.826881e-02,0.211616,0.229885,2002-01-07,99.0
8,SR,2002-01-09,44.764398,287.5,-2.818391e-19,-7.078272e-20,-1.435735e-15,-2.994858e-16,-2.374859e-19,-4.096852e-20,...,5.339772e-16,-1.722987e-19,-3.068236e-19,-5.334981e-16,0.0,4.613563e-05,0.000653,0.000699,2002-01-08,100.0
9,SR,2002-01-10,44.764398,287.5,-1.204602e-15,-5.149841e-17,-2.431526e-12,-7.075751e-14,-1.022604e-15,-4.043747e-17,...,1.083536e-12,-6.010917e-16,-5.413784e-16,-1.082393e-12,0.0,9.361750e-02,0.988595,1.082213,2002-01-09,100.0


In [12]:
#Read in the new precip files
precip_result_dir = Path('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Timeseries_atm_h2/')
precip_files_to_open = ['FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.PRECC.nc',
                           'FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.Timeseries.PRECL.nc']

# Concatenate directory path with each file name separately
file_paths = [precip_result_dir / file_name for file_name in precip_files_to_open]

# Open multiple netCDF files as a single dataset
nc_precip = xr.open_mfdataset(file_paths, combine='nested') # units are in m/s
nc_precip['PRECC_mm'] = nc_precip['PRECC']* (86400*1000) # seconds to days and m to mm = mm/d
nc_precip['PRECL_mm'] = nc_precip['PRECL']* (86400*1000)
nc_precip['PREC_tot_mm'] = nc_precip['PRECC_mm']  + nc_precip['PRECL_mm']

# Only select the cells in .nc that correspond to an NADP site lat/long
subset_list = []
progress_bar = tqdm(total=len(doc_bc_sites))
for index, row in doc_bc_sites.iterrows():
    lat = row['latitude']
    lon = 360-(row['longitude']*-1)   # longitude is positive and based on 360 degrees.
    lat_idx = find_index(nc_precip['lat'].values, lat)   # right now we are doing a 'find nearest' calculation, should probably interpolate across grid cell and have exact coordinates represented?
    lon_idx = find_index(nc_precip['lon'].values, lon)
    subset = nc_precip.isel(lat=lat_idx, lon=lon_idx)
    subset['siteId'] = row['siteId']  # Add 'siteId' as a new coordinate/index
    subset = subset.assign_coords(siteId=row['siteId'])
    subset_list.append(subset)
    progress_bar.update(1)
progress_bar.close()

# Concatenate the list of subsets into a new xarray dataset
nc_precip_nadp = xr.concat(subset_list, dim='siteId')

#Convert to dataframe and assign new time
mod_nadp_precip = nc_precip_nadp.to_dataframe().reset_index()
mod_nadp_precip = mod_nadp_precip.rename(columns = {'time': 'model_time'})

#IMPORTANT STEP: change the time to one day prior (model writes time at end of current day)
mod_nadp_precip['time'] = pd.to_datetime(mod_nadp_precip['model_time'],format='%Y-%m-%d')+DateOffset(days=-1)
mod_nadp_precip = mod_nadp_precip[['siteId', 'time', 'PREC_tot_mm', 'PRECC_mm', 'PRECL_mm']]
mod_nadp_precip.head(10)

100%|██████████| 214/214 [00:00<00:00, 461.60it/s]


,siteId,time,PREC_tot_mm,PRECC_mm,PRECL_mm
0,SR,2001-12-31,0.044211,0.0,0.044211
1,SR,2002-01-01,0.076566,0.0,0.076566
2,SR,2002-01-02,0.029301,0.0,0.029301
3,SR,2002-01-03,0.000037,0.0,0.000037
4,SR,2002-01-04,0.142013,0.0,0.142013
5,SR,2002-01-05,0.302083,0.0,0.302083
6,SR,2002-01-06,2.778388,0.0,2.778388
7,SR,2002-01-07,6.432453,0.0,6.432453
8,SR,2002-01-08,0.015444,0.0,0.015444
9,SR,2002-01-09,5.390756,0.0,5.390756


In [14]:
#Merge with precip df on siteId and time
mod_nadp = mod_nadp[['siteId', 'model_time', 'lat', 'lon', 'Mod_BC_mgm2', 'Mod_SOA_POM_mgm2', 'Mod_SOA_POM_BC_mgm2', 'time', 'SamplingInt']]
mod_nadp_all = pd.merge(mod_nadp, mod_nadp_precip, on = ['siteId', 'time'])
mod_nadp_all.to_csv('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/Daily_Timeseries_NADPsites/PREC_DOC_BC_wetdep_Daily_SamplingIntAssigned_Allsites.csv')
mod_nadp_all

,siteId,model_time,lat,lon,Mod_BC_mgm2,Mod_SOA_POM_mgm2,Mod_SOA_POM_BC_mgm2,time,SamplingInt,PREC_tot_mm,PRECC_mm,PRECL_mm
0,SR,2002-01-01,44.764398,287.50,3.599695e-03,6.264703e-02,6.624672e-02,2001-12-31,98.0,0.044211,0.000000,0.044211
1,SR,2002-01-02,44.764398,287.50,1.474170e-02,1.912378e-01,2.059795e-01,2002-01-01,98.0,0.076566,0.000000,0.076566
2,SR,2002-01-03,44.764398,287.50,3.081979e-03,5.087772e-02,5.395970e-02,2002-01-02,99.0,0.029301,0.000000,0.029301
3,SR,2002-01-04,44.764398,287.50,6.060697e-06,7.930997e-05,8.537066e-05,2002-01-03,99.0,0.000037,0.000000,0.000037
4,SR,2002-01-05,44.764398,287.50,5.835076e-07,4.881186e-06,5.464693e-06,2002-01-04,99.0,0.142013,0.000000,0.142013
...,...,...,...,...,...,...,...,...,...,...,...,...
1641589,CO91,2022-12-28,37.225131,253.75,1.122772e-07,7.288302e-07,8.411074e-07,2022-12-27,NaN,0.072160,0.000000,0.072160
1641590,CO91,2022-12-29,37.225131,253.75,4.139333e-02,3.811552e-01,4.225486e-01,2022-12-28,NaN,4.842455,0.269395,4.573060
1641591,CO91,2022-12-30,37.225131,253.75,2.103874e-02,1.176262e-01,1.386649e-01,2022-12-29,NaN,1.781676,1.026034,0.755642
1641592,CO91,2022-12-31,37.225131,253.75,1.234228e-02,2.136965e-01,2.260387e-01,2022-12-30,NaN,0.710159,0.000000,0.710159


In [15]:
#Then group by siteId and sampling int and sum variables below.
mod_nadp_sum = mod_nadp_all.groupby(['siteId', 'SamplingInt', 'lat', 'lon'])[['PREC_tot_mm', 'Mod_BC_mgm2', 'Mod_SOA_POM_mgm2', 'Mod_SOA_POM_BC_mgm2']].sum().reset_index()
mod_nadp_sum

mod_nadp_sum = pd.merge(mod_nadp_sum, nadp_df, how = 'left', on = ['siteId', 'SamplingInt'])
mod_nadp_sum.to_csv('/glade/campaign/acom/acom-weather/demurray/FCnudged_f09.mam.Murray.Mar26_01.2002_2023.001.cam/SamplingInt_Timeseries_NADPsites/PREC_DOC_BC_Timeseries.SamplingInt_Summed_pairedNADPsites.csv')
mod_nadp_sum

,siteId,SamplingInt,lat,lon,PREC_tot_mm,Mod_BC_mgm2,Mod_SOA_POM_mgm2,Mod_SOA_POM_BC_mgm2,latitude,longitude,dateOn,dateOff,dateOnround,dateOffround,Variable,Value_mgm2,Precip_mm,IntTime
0,AB32,0.0,57.015707,248.75,13.305403,0.069530,1.766915,1.836444,57.1894,-111.6406,2020-10-28,2020-11-04,2020-10-28,2020-11-04,rBC_mgL,0.005256,5.84,7 days 00:00:00
1,AB32,1.0,57.015707,248.75,19.567823,0.092057,2.177492,2.269548,57.1894,-111.6406,2020-11-04,2020-11-10,2020-11-04,2020-11-10,rBC_mgL,0.037476,10.41,6 days 00:00:00
2,AB32,2.0,57.015707,248.75,10.014785,0.083042,1.194062,1.277104,57.1894,-111.6406,2020-11-17,2020-11-24,2020-11-17,2020-11-24,rBC_mgL,0.054303,7.87,7 days 00:00:00
3,AB34,0.0,56.073298,248.75,14.909351,0.099438,2.556430,2.655868,55.6214,-111.1727,2020-10-28,2020-11-03,2020-10-28,2020-11-03,rBC_mgL,0.010290,6.86,6 days 00:00:00
4,AB34,1.0,56.073298,248.75,15.578649,0.085708,2.024307,2.110015,55.6214,-111.1727,2020-11-03,2020-11-10,2020-11-03,2020-11-10,rBC_mgL,0.039024,16.26,7 days 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3690,WY95,1.0,40.994764,253.75,8.184129,0.116103,1.996511,2.112614,41.3647,-106.2408,2020-11-10,2020-11-17,2020-11-10,2020-11-17,rBC_mgL,0.223210,85.85,7 days 00:00:00
3691,WY95,2.0,40.994764,253.75,0.930431,0.040800,0.512369,0.553169,41.3647,-106.2408,2020-11-17,2020-11-24,2020-11-17,2020-11-24,rBC_mgL,0.228600,15.24,7 days 00:00:00
3692,WY97,0.0,42.879581,251.25,9.818033,0.069184,1.243260,1.312444,42.4944,-108.8320,2020-11-03,2020-11-10,2020-11-03,2020-11-10,rBC_mgL,0.030480,15.24,7 days 00:00:00
3693,WY98,0.0,42.879581,250.00,13.002723,0.120434,2.212393,2.332827,43.2227,-109.9917,2020-11-03,2020-11-10,2020-11-03,2020-11-10,rBC_mgL,0.025900,12.95,7 days 00:00:00
